In [68]:
!pip install --upgrade pip
!pip install python-dotenv
!pip install --upgrade openai
!pip install --upgrade langfuse

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 28.2 MB/s  0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 0.28.0
    Uninstalling openai-0.28.0:
      Successfully uninstalled openai-0.28.0


In [69]:
import os
from openai import OpenAI

client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY")
)


Chat API : OpenAI
Let's start with a direct API calls to OpenAI.

In [ ]:
def get_completion(prompt, model="gpt-5.5"):
    messages = [{"role": "user", "content": prompt}]
    completion = client.chat.completions.create(
        model=model,
        messages=messages
    )

    return completion.choices[0].message.content



In [76]:
get_completion("What is 1+1?")

'2'

In [77]:
customer_email = """
Arrr, I be fuming that me blender lid \
flew off and splattered me kitchen walls \
with smoothie! And to make matters worse,\
the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help \
right now, matey!
"""


In [78]:
style = """American English \
in a calm and respectful tone
"""

In [79]:
prompt = f"""Translate the text \
that is delimited by triple backticks 
into a style that is {style}.
text: ```{customer_email}```
"""

print(prompt)

Translate the text that is delimited by triple backticks 
into a style that is American English in a calm and respectful tone
.
text: ```
Arrr, I be fuming that me blender lid flew off and splattered me kitchen walls with smoothie! And to make matters worse,the warranty don't cover the cost of cleaning up me kitchen. I need yer help right now, matey!
```



In [80]:
response = get_completion(prompt)

In [81]:
response

'I’m frustrated that my blender lid came off and splattered smoothie all over my kitchen walls. To make matters worse, the warranty doesn’t cover the cost of cleaning my kitchen. I would appreciate your help with this as soon as possible.'

Chat API : LangChain
Let's try how we can do the same using LangChain.

In [ ]:
#!pip install --upgrade langchain
#!pip install -U langchain-opena

  Using cached tiktoken-0.13.0-cp310-cp310-macosx_10_12_x86_64.whl.metadata (6.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 29.1 MB/s  0:00:00
Using cached tiktoken-0.13.0-cp310-cp310-macosx_10_12_x86_64.whl (1.0 MB)
  Attempting uninstall: openai
    Found existing installation: openai 3.0.0
    Uninstalling openai-3.0.0:0m╺━━━━━━━━━━━━━━━━━━━ 2/4 [openai]
      Successfully uninstalled openai-3.0.0━━━━━━━━━━━━━━━━━━━ 2/4 [openai]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [langchain-openai][langchain-openai]


In [1]:
from langchain_openai import ChatOpenAI


In [4]:
from dotenv import load_dotenv
import os

load_dotenv()


chat = ChatOpenAI(model="gpt-5-nano", api_key=os.environ.get("OPENAI_API_KEY"), temperature=0.7)
chat

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.5.4', 'langchain': '1.3.15'}}, profile={'max_input_tokens': 272000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x115180e50>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x115a726e0>, root_client=<openai.OpenAI object at 0x115180d60>, root_async_client=<openai.AsyncOpenAI object at 0x115a72620>, model_name='gpt-5-nano', model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True)

Prompt template

In [6]:
template_string = """Translate the text \
that is delimited by triple backticks \
into a style that is {style}. \
text: ```{text}```
"""

In [7]:
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate.from_template(template_string)


In [8]:
prompt_template.template

'Translate the text that is delimited by triple backticks into a style that is {style}. text: ```{text}```\n'

In [9]:
prompt_template.input_variables

['style', 'text']

In [10]:
customer_style = """American English \
in a calm and respectful tone
"""

In [11]:
customer_email = """
Arrr, I be fuming that me blender lid \
flew off and splattered me kitchen walls \
with smoothie! And to make matters worse, \
the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help \
right now, matey!
"""

In [12]:
customer_messages = prompt_template.format(
                    style=customer_style,
                    text=customer_email)

In [13]:
print(type(customer_messages))
print(type(customer_messages[0]))

<class 'str'>
<class 'str'>


In [14]:
print(customer_messages)

Translate the text that is delimited by triple backticks into a style that is American English in a calm and respectful tone
. text: ```
Arrr, I be fuming that me blender lid flew off and splattered me kitchen walls with smoothie! And to make matters worse, the warranty don't cover the cost of cleaning up me kitchen. I need yer help right now, matey!
```



In [15]:
# Call the LLM to translate to the style of the customer message
customer_response = chat.invoke(customer_messages)

In [16]:
print(customer_response.content)

I'm frustrated that the blender lid flew off and splattered smoothie onto my kitchen walls. To make matters worse, the warranty doesn't cover the cost of cleaning up my kitchen. I would really appreciate your help right away.


In [17]:
service_reply = """Hey there customer, \
the warranty does not cover \
cleaning expenses for your kitchen \
because it's your fault that \
you misused your blender \
by forgetting to put the lid on before \
starting the blender. \
Tough luck! See ya!
"""

In [18]:
service_style_pirate = """\
a polite tone \
that speaks in English Pirate\
"""

In [22]:
service_messages = prompt_template.format_prompt(
    style=service_style_pirate,
    text=service_reply)

print(service_messages.text)

Translate the text that is delimited by triple backticks into a style that is a polite tone that speaks in English Pirate. text: ```Hey there customer, the warranty does not cover cleaning expenses for your kitchen because it's your fault that you misused your blender by forgetting to put the lid on before starting the blender. Tough luck! See ya!
```



In [24]:
service_response = chat.invoke(service_messages)
print(service_response.content)

Ahoy there, valued customer. Please be advised that the warranty does not cover cleaning costs for yer kitchen, as the issue arose from improper use—ye forgot to put the lid on before startin' the blender. We regret the trouble and thank ye for yer understanding. Fair winds to ye.


## Output Parsers

Let's start with defining how we would like the LLM output to look like:

In [25]:
{
  "gift": False,
  "delivery_days": 5,
  "price_value": "pretty affordable!"
}

{'gift': False, 'delivery_days': 5, 'price_value': 'pretty affordable!'}

In [26]:
customer_review = """\
This leaf blower is pretty amazing.  It has four settings:\
candle blower, gentle breeze, windy city, and tornado. \
It arrived in two days, just in time for my wife's \
anniversary present. \
I think my wife liked it so much she was speechless. \
So far I've been the only one using it, and I've been \
using it every other morning to clear the leaves on our lawn. \
It's slightly more expensive than the other leaf blowers \
out there, but I think it's worth it for the extra features.
"""

review_template = """\
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? \
Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the product \
to arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price,\
and output them as a comma separated Python list.

Format the output as JSON with the following keys:
gift
delivery_days
price_value

text: {text}
"""

In [27]:
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate.from_template(review_template)
print(prompt_template)

input_variables=['text'] input_types={} partial_variables={} template='For the following text, extract the following information:\n\ngift: Was the item purchased as a gift for someone else? Answer True if yes, False if not or unknown.\n\ndelivery_days: How many days did it take for the product to arrive? If this information is not found, output -1.\n\nprice_value: Extract any sentences about the value or price,and output them as a comma separated Python list.\n\nFormat the output as JSON with the following keys:\ngift\ndelivery_days\nprice_value\n\ntext: {text}\n'


In [38]:

llm_model ="gpt-5.5"

messages = prompt_template.format_prompt(text=customer_review)
chat = ChatOpenAI(temperature=0.0, model=llm_model)
response = chat.invoke(messages)
print(response.content)

{
  "gift": true,
  "delivery_days": 2,
  "price_value": [
    "It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features."
  ]
}


In [51]:
import json

response_dict = json.loads(response.content)
type(response_dict)

dict

In [56]:
print(response_dict.keys())
print(response_dict.get('gift'))

dict_keys(['gift', 'delivery_days', 'price_value'])
True
